# OpenFF small molecule parametrization

##### M. Loev

#### Imports

In [10]:
from pathlib import Path

import rdkit
import numpy as np
from openff.toolkit import  Molecule, ForceField, Quantity
from openff.interchange import Interchange


#### Входные данные

`sdf_file` <- path to the SDF file  

`output_dir` <- output directory where all parametrized molecules will be kept 

`forcefield` <- as an argument for the `ForceField()` from one of the available [OpenFF](https://openforcefield.org/force-fields/force-fields/) force fields 

Finally, just press `Run All`

In [11]:
sdf_file = "./example.sdf"
output_dir = "./output/"
forcefield = ForceField("openff-2.3.0.offxml")

#### Функции

In general program works as follows:   

1.`rdkit.Chem.SDMolSupplier()` parses SDF file with structures   


2. For every structure these functions are run in cycle  


3. `obtain_name(rdkit_mol, mol_counter)` obtains molecule name from (`rdkit_mol`). First, the user's "Molecule Name" field is checked, and then, if there is no name, the name in the header is checked using the "_Name" field. If for some reason there is nothing in it (either the field is missing or empty), the name of the type mol<number of the molecule in the SDF file> (`mol_counter`) is assigned to the molecule

4. `make_dir(dir_path, mol_counter)` using the received name, creates a folder in the directory specified for output (`dir_path` contains the path to the output directory, connected to the name of the molecule). If there are more molecules with the same name in the SDF file, a folder is created with the name of the type <name of the molecule>_<number of the molecule in the SDF file> ('mol_counter`). This is less elegant than creating a folder <name of the molecule>_2/3/4/..., but it does not require tracking duplicates for each molecule separately.

5. Next in `create_interchange(rdkit_mol, forcefield)`the RDKit molecule (`rdkit_mol`) is translated into the openff-toolkit object Molecule. On its basis, together with the forcefield force field, a topology is created and an Interchange object is formed. The function returns the Interchange object. 

6. `check_rmsd(rdkit_mol, inter)` checks whether the coordinates in the received Interchange (`inter`) object match the coordinates in the RDKit molecule (`rdkit_mol`). Then, based on these data, the total RMSD for all the molecules contained in the SDF file will be calculated to demonstrate that the coordinates have been fully preserved.

7. Finally, `create_top_gro(molname, interchange, outpath, decimal)` creates two files in the folder (`outpath`) obtained in step 4: topology and coordinates (I did .gro), named in accordance with the name of the molecule obtained in paragraph 1 (`molname`). To maintain the accuracy of the initial coordinates during the transition from angstroms to nanometers , which are characteristic ofI have set the parameter `decimal=5` for the gro files (you can configure it via `decimal`). A cubic cell is pre-created. ([зачем](https://docs.openforcefield.org/projects/interchange/en/stable/using/edges.html#some-mm-engines-do-not-support-no-cutoff-non-bonded-methods))

8. Все эти функции выполняются в sdf_to_gromacs теле основной функции `(rdkit_mol, mol_counter, силовое поле, оказалось = 5, outdir = ".")`

In [12]:
def obtain_name (rdkit_mol, mol_counter):
    
    try:
      mol_name = rdkit_mol.GetProp("Molecule Name")
    except KeyError: 
      if rdkit_mol.HasProp("_Name") and rdkit_mol.GetProp("_Name").strip() != "":
        mol_name = rdkit_mol.GetProp("_Name")
      else: 
        mol_name = f"mol{mol_counter}"

    return mol_name

def make_dir (dir_path, mol_counter):

    try: 
        Path.mkdir(dir_path)
    except FileExistsError: 
        dir_path =dir_path.with_stem(f"{dir_path.stem}_{mol_counter}")
        Path.mkdir(dir_path) 

    return None 


def create_interchange (rdkit_mol, forcefield):

    molecule = Molecule.from_rdkit(rdkit_mol)
    topology = molecule.to_topology()
    inter = Interchange.from_smirnoff(force_field=forcefield,topology= topology)

    return inter


def create_top_gro (molname, interchange, outpath, decim):

    interchange.box = Quantity([10, 10, 10], "nanometer")
    interchange.to_top(outpath / f"{molname}.top")
    interchange.to_gro(outpath/ f"{molname}.gro", decimal = decim)

    return None


def check_coords (rdkit_mol, inter):
    rdkit_coords = rdkit_mol.GetConformer().GetPositions()

    # OpenFF координаты → переводим в Å
    off_coords = inter.positions.m_as("angstrom")

    # RMSD
    diff = rdkit_coords - off_coords
    rmsd = np.sqrt((diff ** 2).sum(axis=1).mean())
    
    return rmsd


def sdf_to_gromacs (rdkit_mol, mol_counter, forcefield, decim = 5, outdir = "."):

    mol_name = obtain_name(rdkit_mol, mol_counter)

    out_path = Path(outdir) / mol_name
    make_dir(out_path, mol_counter)

    inter = create_interchange(rdkit_mol, forcefield)
    
    rmsd = check_coords(rdkit_mol, inter)

    create_top_gro(mol_name, inter, out_path, decim)

    
    return rmsd


#### Обработка SDF файла

The first step is to create (if necessary) a folder in which all the results will be stored.

I work in the context manager for security. In 'rdkit.Chem.SDMolSupplier()`, I write (in addition to the path to the SDF file `sdf_file`) `removeHs=False` to save the hydrogen atoms from the SDF file.

`mol_counter` - counter of molecules in SDF file

`errors` - a counter of molecules that have not been processed for one reason or another

`rmsd` - total RMSD count

In [13]:
Path(output_dir).mkdir(parents=True, exist_ok=True)

with rdkit.Chem.SDMolSupplier(sdf_file, removeHs = False) as suppl:
  mol_counter = 0
  errors = 0
  rmsd = 0

  for mol in suppl:
    mol_counter+=1
    if mol is None:
      errors+=1
      continue
    
    rmsd += sdf_to_gromacs(mol, mol_counter, forcefield, outdir=output_dir)

  print(f"Processed {mol_counter} molecules")
  print(f"Failed to process {errors} molecules")
  print(f"Total RMSD of coordinates is {rmsd} Å")

[02:13:22] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
/home/domain/mlmisha/miniconda3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
[02:13:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
/home/domain/mlmisha/miniconda3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
[02:13:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero.

Processed 54 molecules
Failed to process 0 molecules
Total RMSD of coordinates is 3.093557549836968e-14 Å
